# 📘 부록: OKF 비서 만들기 2. 코딩 에이전트와 스킬로 위키 키우기

부록 02의 비서는 위키를 읽고 답만 했습니다. 이번에는 위키를 키웁니다. 새 자료가 오면 위키에 넣고, 위키를 점검하는 일을 Claude Code와 Codex가 스킬로 합니다. 스킬을 구성하고, 두 도구가 찾는 폴더에 등록한 뒤, 터미널에서 지시해 활용합니다. 두 코딩 에이전트는 등록된 `SKILL.md`를 스스로 찾아 읽고, 위키 파일을 직접 고치고, 검사 스크립트까지 실행합니다.

<img src="images/14_agent_skills_ingest.png" width="1050" alt="사람이 새 자료를 넣으라고 지시하면 에이전트가 스킬 okf-ingest를 고르고, raw/의 원본은 읽기만 한 뒤 위키의 문서, 목차, 변경 기록을 고치고 check.py로 검사합니다. Claude Code는 .claude/skills/, Codex는 .agents/skills/에서 같은 SKILL.md를 찾습니다.">

**공식 문서:** [Claude Code 스킬](https://code.claude.com/docs/en/skills), [Codex 스킬](https://developers.openai.com/codex/skills)

**학습 목표**

1. **스킬 폴더(`SKILL.md`와 스크립트)를 구성합니다.**
2. **같은 스킬을 Claude Code와 Codex가 찾는 폴더에 등록합니다.**
3. **등록한 스킬로 새 자료를 위키에 넣고 점검하게 한 뒤, 바뀐 줄과 검사 결과를 확인합니다.**

## 1. 스킬 구성

### 1-1. 스킬과 점진적 공개

**스킬(skill)** 은 에이전트가 특정 일을 할 때 펼쳐 보는 업무 설명서입니다. 폴더 하나에 `SKILL.md`와, 그 절차가 쓰는 스크립트나 자료를 담습니다. `SKILL.md`는 맨 위 frontmatter(`name`, `description`)와 본문(절차, 규칙)으로 이루어집니다. 이 형식(Agent Skills)은 Claude Code, Codex 같은 코딩 에이전트가 그대로 읽습니다.

<img src="images/15_agent_skills_disclosure.png" width="1050" alt="코딩 에이전트는 처음에 스킬의 이름과 설명만 알고, 일을 맡으면 SKILL.md 본문을 펼치며, 본문이 가리키는 스크립트 check.py는 실행할 때만 씁니다.">

- **1단계, 처음부터 아는 것:** 스킬마다 이름과 한 줄 설명만 압니다.
- **2단계, 필요할 때 펼침:** 지시가 설명과 맞으면 그 스킬의 `SKILL.md` 본문을 읽습니다.
- **3단계, 지침이 가리키는 것:** 본문이 가리키는 스크립트는 실행할 때만 씁니다. 스크립트 코드를 읽지 않아도 실행 결과만으로 쓸 수 있습니다.

스킬이 열 개, 스무 개로 늘어도 처음 맥락에는 설명 한 줄씩만 들어갑니다. 도구가 동작 하나라면, 스킬은 그 도구들을 어떤 순서와 규칙으로 쓸지 적은 지침입니다. LLM 위키의 세 층 가운데 스키마(위키 규칙 문서)에 해당합니다.

### 1-2. 스킬 폴더의 구성

스킬 하나는 폴더 하나입니다. 폴더 안에 `SKILL.md`와, 그 절차가 쓰는 스크립트를 둡니다. 이 부록의 스킬 두 개는 작업 폴더 `agent_wiki/`의 스킬 폴더에 있습니다(2절).

~~~text
agent_wiki/.claude/skills/
├── okf-ingest/
│   ├── SKILL.md            새 자료 하나를 위키에 넣는 절차와 쓰기 규칙
│   └── scripts/check.py    검사 스크립트
└── okf-lint/
    ├── SKILL.md            위키를 점검해 보고하는 절차(문서는 고치지 않음)
    └── scripts/check.py    같은 검사 스크립트
~~~

- **frontmatter:** `name`(스킬 이름), `description`(언제 쓰는 스킬인지), `allowed-tools`(Claude Code가 묻지 않고 쓸 도구)입니다. 1단계에서 에이전트가 아는 것이 이 칸입니다.
- **본문:** 순서와 규칙입니다. 에이전트가 그대로 따릅니다.
- **스크립트:** 본문이 실행하라고 한 파일입니다. 에이전트는 코드를 읽지 않아도 실행 결과만으로 씁니다.

### 1-3. 넣기 스킬

넣기 스킬은 자료 한 건을 네 갈래 가운데 하나로 위키에 넣습니다. 위에서부터 차례로 보고 처음 맞는 갈래를 고릅니다.

| 갈래 | 언제 | 할 일 |
|---|---|---|
| 기존 문서 수정 | 자료의 사실이 기존 문서에 이미 있는 절(제목)에 들어갈 때 | 그 문서를 고칩니다. 대상이 더는 현재가 아니면 `deprecated`로 바꿉니다 |
| 새 문서 | 맞는 폴더는 있는데 자료가 다루는 대상의 문서가 없을 때 | 그 폴더에 문서를 만들고 폴더 목차에 한 줄 더합니다 |
| 새 폴더 | 번들의 주제에는 맞는데 맞는 폴더가 없을 때 | 폴더와 폴더 목차를 만들고 맨 위 목차에 한 줄 더합니다 |
| 새 번들 | 어느 번들의 폴더 설명에도 맞지 않을 때 | 맨 위 목차, 변경 기록, 폴더, 문서를 갖춘 번들을 만듭니다 |

광고처럼 남길 필요가 없는 자료는 건너뜁니다. 번들과 폴더는 목차의 폴더 설명을 자료의 주제와 견줘 고르고, 문서 안에 같은 이름이 나온다는 이유만으로 고르지 않습니다. 기존 문서에 새 절을 만들어야 넣을 수 있다면 그 문서의 대상이 아니므로 다음 갈래로 내려갑니다.

<img src="images/18_ingest_four_branches.png" width="1050" alt="넣을 곳 고르기: 새 자료가 오면 맞는 번들이 있는지, 맞는 폴더가 있는지, 대상 문서가 있는지를 차례로 묻습니다. 아니오이면 차례로 새 번들, 새 폴더, 새 문서이고, 모두 예이면 기존 문서 수정입니다. 남길 필요가 없으면 건너뜁니다.">

갈래를 정한 뒤에는 아래 순서로 문서를 쓰고, 목차와 변경 기록을 고치고, 검사합니다. 답에는 결과와 함께 사람이 확인할 것(새 번들이나 새 폴더, `deprecated` 판단, `verified`를 지운 문서, 판단으로 정한 값, 어긋날 수 있는 다른 문서)을 적습니다. 쓰기 규칙은 모델이 실제로 틀렸던 것(자료에 없는 날짜를 붙임, 이력을 본문에 씀)을 막도록 적었습니다.

<img src="images/16_agent_ingest_steps.png" width="1050" alt="넣기 스킬 okf-ingest의 여덟 단계: 1 자료 읽기(raw/ 읽기만), 2 번들 고르기(맨 위 목차, 광고면 건너뜀, 맞는 번들이 없으면 새 번들), 3 문서 고르기(폴더 목차, 새 문서나 새 폴더), 4 문서 쓰기(쓰기 규칙, frontmatter), 5 목차에 한 줄(새 문서일 때), 6 변경 기록(log.md 한 줄), 7 검사(check.py, 문제 있으면 4로 돌아가 고쳐 다시), 8 답하기(결과, 사람이 확인할 것).">

에이전트가 일을 맡으면 펼쳐 읽는 넣기 스킬의 본문입니다. 위 순서와 규칙이 모두 여기에 있습니다.

`agent_wiki/.claude/skills/okf-ingest/SKILL.md`

~~~markdown
---
name: okf-ingest
description: raw/의 새 자료 하나를 OKF 위키(bundles/)에 넣는다. 사람이 새 자료를 위키에 넣으라고 할 때 쓴다.
allowed-tools: Read, Edit, Write, Glob, Bash(python3 *)
---
# OKF 위키에 새 자료 넣기

작업 폴더의 AGENTS.md 규칙을 따른다. 오늘 날짜와 지금 시각은 `date` 명령으로 확인한다. 스크립트는 이 SKILL.md 옆 `scripts/check.py`다. 작업 폴더에서 Claude Code는 `python3 .claude/skills/okf-ingest/scripts/check.py`, Codex는 `python3 .agents/skills/okf-ingest/scripts/check.py`로 실행한다.

## 넣을 곳: 네 갈래

자료를 넣는 방법은 네 갈래다. 위에서부터 차례로 보고 처음 맞는 갈래를 고른다. 광고, 스팸처럼 남길 필요가 없는 자료는 넣지 않고 건너뛴다.

| 갈래 | 언제 | 할 일 |
|---|---|---|
| 기존 문서 수정 | 자료가 기존 문서의 사실을 바꾸거나 더할 때 | 그 문서를 고친다. 대상이 더는 현재가 아니면 `deprecated`로 바꾼다 |
| 새 문서 | 맞는 폴더는 있는데 자료가 다루는 대상의 문서가 없을 때 | 그 폴더에 문서를 만들고 폴더 `index.md`에 한 줄 더한다 |
| 새 폴더 | 번들의 주제에는 맞는데 맞는 폴더가 없을 때 | 아래 '새 폴더 만들기' |
| 새 번들 | 어느 번들의 폴더 설명에도 맞지 않을 때 | 아래 '새 번들 만들기' |

번들과 폴더는 맨 위 `index.md`와 폴더 `index.md`의 설명을 자료의 주제와 견줘 고른다. 문서 안에 같은 이름(게임, 사람, 프로젝트 이름)이 나온다는 이유만으로 그 번들이나 문서를 고르지 않는다. 기존 문서 수정은 자료의 사실이 그 문서에 이미 있는 절(제목) 가운데 하나에 들어갈 때만 고른다. 들어갈 절이 없어 새 절을 만들어야 한다면 그 자료는 그 문서의 대상이 아니므로, 기존 문서를 고치지 않고 다음 갈래(새 문서, 새 폴더, 새 번들)로 내려간다. 회의 메모라면 회의 정리 문서가 아니라 그 결정으로 내용이 바뀌는 문서를 고친다.

## 순서

1. 자료(`raw/<파일 이름>`)를 읽는다. 자료는 고치지 않는다.
2. 번들 고르기: 번들마다 맨 위 `index.md`를 읽고 자료를 넣을 번들을 고른다. 남길 필요가 없으면 "결과: 건너뜀"과 이유를 답하고 끝낸다. 맞는 번들이 없으면 새 번들 갈래다.
3. 문서 고르기: 고른 번들의 폴더 `index.md`를 읽고 기존 문서 수정, 새 문서, 새 폴더 가운데 하나를 고른다.
4. 문서 쓰기: 고치는 문서면 원래 문서를 읽고, 아래 쓰기 규칙과 frontmatter 규칙대로 쓴다.
5. 목차: 새 문서는 폴더 `index.md`에 `* [제목](파일.md) - 한 줄 설명` 한 줄을 더한다. 새 폴더, 새 번들은 각 '만들기' 절대로 목차를 만든다.
6. 변경 기록: 번들 `log.md`의 오늘 날짜 제목(`## YYYY-MM-DD`) 아래(없으면 맨 위 제목 바로 아래에 만든다)에 한 줄을 적는다. 고친 문서는 `* **Update**: <자료 파일 이름>를 반영해 [제목](/폴더/파일.md) 문서를 고쳤습니다.`, 새 문서는 `* **Creation**: <자료 파일 이름>를 반영해 [제목](/폴더/파일.md) 문서를 만들었습니다.`
7. check.py로 검사하고, 문제가 나오면 고친 뒤 다시 검사한다.
8. 한국어로 답한다. 첫 줄은 `결과:` 뒤에 갈래(기존 문서 수정, 새 문서, 새 폴더, 새 번들, 건너뜀), 번들/개념 ID, 상태다. 그 아래 `사람이 확인할 것:` 목록을 적는다. 확인할 것이 없으면 `사람이 확인할 것: 없음`이라고 적는다.

## 사람이 확인할 것

에이전트가 판단으로 정했거나, 사람의 검토가 사라졌거나, 다른 문서와 어긋날 수 있는 곳을 빠짐없이 알린다.

- 새 번들, 새 폴더를 만들었으면 그 이름과 위치.
- `status`를 `deprecated`로 바꿨으면 그 판단과 근거 문장.
- `verified`를 지운 문서(사람 검토 기록이 사라져 검증 전이 됨).
- 자료가 애매해 판단으로 정한 값이나 문장(예: 연도가 없는 날짜, 누가 맡는지 불분명한 일).
- 바꾼 값과 같은 대상을 다르게 적은 다른 문서(예: 같은 날짜를 적은 체크리스트). 그 문서는 고치지 않고 알리기만 한다.

## 쓰기 규칙

- 고치는 문서면 원래 본문의 구조와 내용을 유지하고, 자료가 바꾼 사실만 고치거나 더한다. title은 그대로 두고, description은 자료 때문에 틀리게 된 사실만 고친다. description은 한 줄 설명이라 각주 표시를 붙이지 않는다.
- 새 문서면 같은 폴더 문서들의 양식(제목 단계, 목록 모양)을 따른다.
- 다른 문서는 `[이름](/폴더/파일.md)` 꼴로 링크한다. 링크는 번들 맨 위 기준이라 번들 이름은 넣지 않는다(`/people/jiho.md`는 맞고 `/studio/people/jiho.md`는 끊긴다). 태그는 영문 소문자로 쓴다.
- 자료에서 온 사실 뒤에는 각주 표시 `[^자료 파일 이름에서 확장자를 뺀 것]`을 붙이고, 바뀐 사실에 붙어 있던 옛 각주 표시는 지운다. 본문 맨 끝에 각주 정의 줄 `[^각주 이름]: 자료 파일 이름`을 둔다.
- 바뀐 값은 새 값만 적는다(예: '정식 출시: 12월 4일'). '무엇에서 무엇으로 바뀜' 같은 이력은 본문과 description에 쓰지 않는다. 이력은 log.md에 남는다.
- 자료에 없는 연도, 날짜, 숫자를 붙이지 않는다. 바꾸지 않는 문장은 글자 그대로 둔다. 자료에 없는 사실은 지어내지 않는다. 한국어로 쓴다.

## 새 폴더 만들기

- 폴더: `bundles/<번들>/<영문 소문자 이름>/`(예: `support`).
- 폴더 `index.md`: `# 폴더 이름`과 문서 목록(`* [제목](파일.md) - 한 줄 설명`). frontmatter는 넣지 않는다.
- 번들 맨 위 `index.md`: 폴더 목록에 `* [폴더 이름](폴더/) - 한 줄 설명` 한 줄을 더한다. 맨 위 목차의 frontmatter(`okf_version`)는 그대로 둔다.
- 새 문서의 `type`은 대상에 맞게 정한다. 변경 기록에는 6번대로 Creation 한 줄을 적는다.

## 새 번들 만들기

- 폴더: `bundles/<영문 소문자 이름>/`(예: `support`).
- 맨 위 `index.md`: frontmatter는 `okf_version: "0.2"` 하나만 두고, 본문은 `# 위키 제목`과 폴더마다 `* [폴더 이름](폴더/) - 한 줄 설명`이다.
- 폴더: 자료의 대상에 맞는 폴더 하나를 만들고, 그 안에 폴더 `index.md`(`# 폴더 이름`과 문서 목록, frontmatter 없음)와 새 문서를 둔다. 새 문서의 `type`은 대상에 맞게 정한다.
- `log.md`: `# 위키 제목 변경 기록`, 오늘 날짜 제목 `## YYYY-MM-DD`, 그 아래 `* **Initialization**: 번들을 만들었습니다.`와 Creation 한 줄(순서 6번의 기록을 겸한다).

## frontmatter 규칙

- 고치는 문서: 원래 키는 살린다. `verified`는 지운다(내용이 바뀌었으니 사람이 다시 확인한다). `generated`는 `{ by: "도구 이름", at: "지금 시각(ISO 8601, 예: 2026-11-02T09:00:00+09:00)" }`로 바꾼다(도구 이름은 claude-code 또는 codex). `sources`에 `{ id: 각주 이름, resource: "raw://자료 파일 이름", title: 자료 파일 이름 }` 항목을 더한다. 자료 때문에 대상이 더는 현재가 아니면(종료, 폐기, 퇴사, 해지) `status: deprecated`로 바꾼다.
- 새 문서: `type`은 같은 폴더 문서의 type을 따르고, `title`, `description`, `tags`, `status: draft`, `generated`, `sources`를 적는다. 파일 이름은 영문 소문자와 밑줄로 짓는다.
~~~

### 1-4. 점검 스킬과 검사 스크립트

- **점검 스킬(`okf-lint`):** `check.py`를 실행하고, 오늘 바뀐 문서와 링크로 이어진 문서를 읽어 서로 다른 값을 찾아 표로 보고합니다. 문서는 고치지 않습니다.

<img src="images/17_agent_lint_steps.png" width="1050" alt="점검 스킬 okf-lint의 네 단계: 1 검사 실행(check.py로 적합성, 끊긴 링크, 만료, 없는 원문), 2 오늘 바뀐 문서(log.md의 오늘 날짜), 3 이웃 문서와 대조(링크로 이어진 문서, 회의 정리 문서는 빼고), 4 표로 보고(검사 결과, 모순). 문서는 고치지 않고 사람이 정하게 알립니다.">

- **검사 스크립트(`check.py`):** 적합성(부록 01 6절), 끊긴 링크, 만료, 없는 원문(`raw://`)을 찾습니다. 표준 라이브러리만 써서 어느 파이썬으로도 돕니다.

## 2. 스킬 등록

### 2-1. 도구마다 스킬을 찾는 폴더

스킬을 등록한다는 것은 도구가 찾는 폴더에 스킬 폴더를 두는 것입니다. 두 도구는 찾는 폴더가 다릅니다.

| | Claude Code | Codex |
|---|---|---|
| 스킬 폴더 | 작업 폴더의 `.claude/skills/<이름>/SKILL.md` | 작업 폴더의 `.agents/skills/<이름>/SKILL.md` |
| frontmatter | 모든 키가 선택이고 `description`을 권장합니다 | `name`, `description`이 꼭 있어야 합니다 |
| 직접 부르기 | `/okf-ingest` | `$okf-ingest` |
| 파일 고치기 | 스킬의 `allowed-tools`에 적은 도구는 묻지 않고 쓰고, 그 밖의 도구는 쓸 때 묻습니다 | 처음 실행할 때 이 폴더에서 작업해도 되는지 묻습니다. 허용하면 작업 폴더 안의 파일을 고칩니다 |

그래서 같은 스킬을 두 폴더에 모두 넣습니다. Claude Code만 쓰는 키(`allowed-tools`)는 Codex가 모르는 키라 무시합니다.

### 2-2. 이 부록의 작업 폴더

작업 폴더 `agent_wiki/`는 부록 폴더에 미리 만들어 두었습니다. 규칙 파일, 번들, 원본 자료가 들어 있고, 스킬 두 개가 두 도구의 스킬 폴더에 등록되어 있습니다. 원본 `data/`는 그대로 두고 이 폴더의 위키를 고칩니다.

~~~text
agent_wiki/
├── AGENTS.md                 규칙 파일(부록 02)
├── bundles/                  위키(작업 사본). 에이전트가 고치는 곳
├── raw/                      원본 자료. 읽기만 함
├── .claude/skills/           Claude Code가 찾는 스킬 폴더
│   ├── okf-ingest/
│   └── okf-lint/
└── .agents/skills/           Codex가 찾는 스킬 폴더
    ├── okf-ingest/
    └── okf-lint/
~~~

등록이 됐는지는 도구의 대화 창에서 확인합니다. `agent_wiki` 폴더에서 `claude`를 실행해 `/`를 치면 명령 목록에 `/okf-ingest`, `/okf-lint`가 보이고, `codex`를 실행해 `/skills`를 치면 두 스킬이 보입니다. 실습을 처음부터 다시 하려면 부록 폴더를 다시 받아 `agent_wiki`를 처음 상태로 되돌립니다.

## 3. 스킬 활용

### 3-1. 지시하는 방법과 새 자료 일곱 건

**준비:** Claude Code나 Codex CLI가 설치되어 로그인되어 있어야 합니다(부록 02 1-2). `agent_wiki` 폴더에서 터미널을 열고 `claude`(Claude Code) 또는 `codex`(Codex)를 실행합니다. 폴더를 믿을지, 파일을 고쳐도 될지 묻는 창이 나오면 허용합니다. 스킬이 `python3`로 검사 스크립트를 실행하므로, Windows에서 `python3`가 없으면 `SKILL.md`의 `python3`(frontmatter와 본문)를 `python`으로 바꿔 씁니다.

대화 창에 지시하는 방법은 두 가지입니다.

| 지시 방법 | Claude Code 대화 창 | Codex 대화 창 |
|---|---|---|
| 스킬 직접 부르기 | `/okf-ingest raw/01_meeting_2026-10-31.md` | `$okf-ingest raw/01_meeting_2026-10-31.md` |
| 지시어로 맡기기 | `raw/01_meeting_2026-10-31.md 자료를 OKF 위키에 넣어 줘` | `raw/01_meeting_2026-10-31.md 자료를 OKF 위키에 넣어 줘` |

지시어로 맡기면 에이전트가 스킬의 `description`("새 자료 하나를 OKF 위키에 넣는다")과 지시를 견줘 `okf-ingest`를 스스로 고릅니다. 직접 부르면 고르는 단계 없이 바로 그 스킬을 씁니다. Claude Code는 지시어로 맡기면 스킬의 `allowed-tools`가 적용되지 않아, 파일을 고치거나 검사 스크립트를 실행할 때 허락을 묻습니다. 허용하면 이어서 진행합니다.

`raw/`에는 새 자료 일곱 건이 있고, 자료마다 위키에 들어가는 갈래가 다릅니다. 아래 절마다 한 갈래씩 넣어 봅니다.

| 자료 | 내용 | 갈래 | 절 |
|---|---|---|---|
| `01_meeting_2026-10-31.md` | 긴급 회의 메모: 심사 제출일과 출시일 연기 | 기존 문서 수정 | 3-2 |
| `02_mail_new_member.txt` | 새 팀원(사운드 디자이너) 소개 메일 | 새 문서(프로젝트 문서도 수정) | 3-3 |
| `03_data_memo_arppu.md` | 새 지표 ARPPU 추가 메모 | 새 문서 | 3-3 |
| `04_data_memo_conversion.md` | 결제 전환율 지표 중단 메모 | 기존 문서 수정(`deprecated`) | 3-3 |
| `05_ad_mail.txt` | 광고 메일 | 건너뜀 | 3-3 |
| `07_equipment_memo.md` | 사내 장비 목록 메모 | 새 폴더 | 3-4 |
| `06_lore_memo.md` | 게임 세계관 메모 | 새 번들 | 3-5 |

아래 **⌨️ 입력**은 대화 창에 치는 지시이고, **💬 답**과 바뀐 줄은 이 부록을 만들 때 Codex에 일곱 건을 차례로 넣은 실제 기록입니다. Claude Code는 입력의 `$`를 `/`로 바꿔 칩니다. 같은 스킬을 읽으므로 같은 순서로 일하지만, 문장은 도구와 실행마다 조금씩 다릅니다.

### 3-2. 기존 문서 수정

`raw/01_meeting_2026-10-31.md`는 심사 제출일과 정식 출시일을 미룬 회의 메모입니다. 바뀐 날짜는 프로젝트 문서에 이미 있는 `# 일정` 절에 들어가므로 그 문서를 고칩니다.

⌨️ 입력 (Codex, 스킬 직접 부르기)

~~~text
$okf-ingest raw/01_meeting_2026-10-31.md
~~~

💬 답 (Codex)

~~~text
결과: 기존 문서 수정, studio/projects/chorong_launch (stable)

사람이 확인할 것:

- `studio/playbooks/launch_checklist`에는 출시일이 아직 11월 20일로 적혀 있습니다. 이번 반영에서는 그 문서를 수정하지 않았습니다.
- 10월 27일 회의록에는 당시 심사일과 출시일이 남아 있습니다. 과거 기록으로 유지했습니다.
- 검사에서 별도 기존 문서 `studio/playbooks/server_incident`의 `stale_after: 2026-10-01` 만료가 보고됐습니다. 이번 자료와 무관해 수정하지 않았습니다.
~~~

바뀐 줄입니다(`-`는 지운 줄, `+`는 더한 줄). 직접 해 본 뒤에는 작업 폴더의 `bundles/studio/projects/chorong_launch.md`와 `bundles/studio/log.md`를 열어 같은 곳이 바뀌었는지 봅니다.

~~~diff
===== studio/projects/chorong_launch.md
-generated: { by: "starpixel-secretary/0.1", at: "2026-10-27T18:00:00+09:00" }
-verified: { by: "human:yuna", at: "2026-10-28T09:30:00+09:00" }
+  - id: 01_meeting_2026-10-31
+    resource: raw://01_meeting_2026-10-31.md
+    title: 01_meeting_2026-10-31.md
+generated: { by: "codex", at: "2026-11-02T17:14:55+09:00" }
-* 스토어 심사 제출: 11월 6일[^weekly-1027]
-* 정식 출시: 11월 20일[^weekly-1027]
+* 스토어 심사 제출: 11월 13일[^01_meeting_2026-10-31]
+* 정식 출시: 12월 4일[^01_meeting_2026-10-31]
-출시 전 남은 항목은 [출시 체크리스트](/playbooks/launch_checklist.md)에서 관리합니다.
+출시 전 남은 항목은 [출시 체크리스트](/playbooks/launch_checklist.md)에서 관리합니다. 박서연은 환불 오류 수정을 11월 10일까지 마칩니다.[^01_meeting_2026-10-31]
-[^weekly-1027]: 10월 27일 주간 회의
+[^01_meeting_2026-10-31]: 01_meeting_2026-10-31.md

===== studio/log.md
+
+## 2026-11-02
+* **Update**: 01_meeting_2026-10-31.md를 반영해 [초롱 원정대 출시](/projects/chorong_launch.md) 문서를 고쳤습니다.
~~~

**결과 읽기**
- **고친 문서:** `# 일정` 절의 두 날짜가 새 값으로 바뀌고, 각주가 새 자료(`[^01_meeting_2026-10-31]`)를 가리킵니다. 박서연의 수정 기한은 이미 있는 `# 남은 일` 절에 더해집니다.
- **frontmatter:** `verified`가 지워지고(내용이 바뀌어 사람이 다시 확인해야 함), `generated`가 도구 이름과 지금 시각으로, `sources`에 `raw://` 출처가 더해집니다.
- **변경 기록:** 번들 `log.md`에 오늘 날짜 제목(`## YYYY-MM-DD`)이 생기고 Update 한 줄이 적힙니다.
- **사람이 확인할 것:** 출시 체크리스트에는 출시일이 아직 11월 20일로 남아 있다고 알립니다. 스킬이 다른 문서는 고치지 않고 알리기만 하라고 했기 때문입니다. 이 어긋남은 3-6 점검에서 다시 나옵니다.

### 3-3. 새 문서, deprecated, 건너뜀

새 팀원 메일과 ARPPU 메모는 맞는 폴더(사람, 지표)는 있는데 대상의 문서가 없어 새 문서가 됩니다. 결제 전환율 메모는 기존 지표 문서를 고쳐 `deprecated`로 바꾸고, 광고 메일은 건너뜁니다.

⌨️ 입력 (Codex). 한 줄씩 차례로 칩니다.

~~~text
$okf-ingest raw/02_mail_new_member.txt
$okf-ingest raw/03_data_memo_arppu.md
$okf-ingest raw/04_data_memo_conversion.md
$okf-ingest raw/05_ad_mail.txt
~~~

💬 답 (Codex)

~~~text
[raw/02_mail_new_member.txt]
결과: 새 문서 + 기존 문서 수정, `studio/people/eunchae`, `studio/projects/chorong_launch`, 완료

목차와 변경 기록도 갱신했습니다. 검사에서는 기존 `studio/playbooks/server_incident.md`의 만료된 `stale_after` 경고가 보고됐습니다. 이번 자료와 무관해 수정하지 않았습니다.

사람이 확인할 것:

- 원문에 합류 연도가 없어 `11월 3일`로 기록했습니다. 연도를 확인해 주세요.

[raw/03_data_memo_arppu.md]
결과: 새 문서, `game_data/metrics/arppu`, draft

ARPPU 정의와 계산식을 기록하고 지표 목차와 변경 기록을 갱신했습니다. 검사기는 새 문서 문제를 찾지 않았고, 기존 문서에서 만료 표시 1건을 보고했습니다.

사람이 확인할 것:

- 원문에 날짜가 없어 “다음 주”의 기준 시점을 알 수 없습니다. 대시보드 계산이 아직 계획 단계인지 확인해 주세요.
- 검사기에서 기존 `studio/playbooks/server_incident.md`의 `stale_after: 2026-10-01` 만료를 확인했습니다.

[raw/04_data_memo_conversion.md]
결과: 기존 문서 수정, `game_data/metrics/paying_conversion`, `deprecated` 반영 완료.

분모 불일치와 주간 보고·대시보드 제외 내용을 결제 전환율 문서(bundles/game_data/metrics/paying_conversion.md)에 반영하고, 변경 기록(bundles/game_data/log.md)을 추가했습니다.

사람이 확인할 것:

- 자료의 “쓰지 않기로 했습니다”를 지표의 `deprecated` 상태로 해석했습니다.
- 검사에서 이번 변경과 무관한 `studio/playbooks/server_incident.md`의 `stale_after: 2026-10-01` 항목이 확인됐습니다. 수정하지 않았습니다.

[raw/05_ad_mail.txt]
결과: 건너뜀, 대상 없음, 처리하지 않음. 광고성 할인 메일이라 위키에 넣지 않았습니다.

사람이 확인할 것: 없음
~~~

결제 전환율 문서의 바뀐 줄입니다.

~~~diff
===== game_data/metrics/paying_conversion.md
-description: 가입한 플레이어 가운데 한 번 이상 결제한 비율.
+description: 정의가 합의되지 않아 사용하지 않기로 한 지표.
-status: draft
-generated: { by: "starpixel-secretary/0.1", at: "2026-10-29T20:00:00+09:00" }
+status: deprecated
+generated: { by: "codex", at: "2026-11-02T17:18:47+09:00" }
+sources:
+  - { id: 04_data_memo_conversion, resource: "raw://04_data_memo_conversion.md", title: "04_data_memo_conversion.md" }
-[players](/tables/players.md) 가운데 [purchases](/tables/purchases.md)에 한 행 이상 있는 플레이어의 비율입니다. 계산 방법과 최근 값을 아직 정하지 않아 초안입니다.
+팀마다 분모를 가입자 전체 또는 접속한 플레이어로 달리 사용해, 같은 이름으로 서로 다른 숫자가 보고되었습니다. 결제 전환율은 사용하지 않으며, 주간 보고와 대시보드에서 제외합니다.[^04_data_memo_conversion]
+
+[^04_data_memo_conversion]: 04_data_memo_conversion.md
~~~

**결과 읽기**
- **새 문서:** 사람 문서(`people/eunchae`)와 지표 문서(`metrics/arppu`)는 `status: draft`로 만들어지고, 각 폴더 `index.md`에 한 줄씩 더해집니다. 새 팀원 메일은 프로젝트 문서에도 트레일러 음악 담당을 더했습니다. 자료 하나가 문서 두 개를 바꿀 수 있습니다.
- **deprecated:** 결제 전환율 문서는 `status`가 `deprecated`로 바뀝니다. 에이전트가 frontmatter 규칙('대상이 더는 현재가 아니면 deprecated')을 읽고 정하고, 답에 그 판단을 알립니다.
- **판단이 들어간 줄:** 원래 정의 문장(`players` 가운데 `purchases`에 한 행 이상 있는 플레이어의 비율)이 지워지고 중단 이유로 바뀌었습니다. 쓰지 않게 된 이유를 남긴 판단이지만 원래 정의는 문서에서 사라졌습니다. 에이전트의 판단이 문서에 바로 들어가므로, 바뀐 줄을 사람이 확인하고 원래 정의를 남길지 정합니다.
- **건너뜀:** 광고 메일은 위키에 아무것도 남기지 않습니다.

### 3-4. 새 폴더

`raw/07_equipment_memo.md`는 사내 장비(테스트폰, 태블릿, 마이크)를 누가 쓰는지 적은 목록입니다. 업무 위키의 주제에는 맞지만 사람, 프로젝트, 회의, 업무 절차, 결정 가운데 맞는 폴더가 없어 새 폴더를 만듭니다.

⌨️ 입력 (Codex)

~~~text
$okf-ingest raw/07_equipment_memo.md
~~~

💬 답 (Codex)

~~~text
결과: 새 폴더, `studio/equipment/inventory`, draft

사람이 확인할 것:

- `bundles/studio/equipment/`를 새로 만들었습니다.
- 메모에 연도가 없어 대여 기한은 “11월 15일까지”로 기록했습니다.
- 검사에서 기존 문서 `server_incident.md`의 `stale_after: 2026-10-01`이 만료된 것으로 확인됐습니다. 이번 작업에서는 수정하지 않았습니다.
~~~

바뀐 줄입니다. 폴더 목차와 장비 문서는 새 파일이라 모든 줄이 `+`입니다.

~~~diff
===== studio/index.md
+* [장비](equipment/) - 사내 장비 보유와 대여 현황

===== studio/equipment/index.md
+# 장비
+
+* [사내 장비 목록](inventory.md) - 사내 테스트폰, 태블릿, 마이크의 보유·사용 현황

===== studio/equipment/inventory.md
+---
+type: Equipment Inventory
+title: 사내 장비 목록
+description: 사내 테스트폰, 태블릿, 마이크의 보유·사용 현황.
+tags: [equipment, inventory]
+status: draft
+generated: { by: "codex", at: "2026-11-02T17:24:12+09:00" }
+sources:
+  - id: 07_equipment_memo
+    resource: raw://07_equipment_memo.md
+    title: 07_equipment_memo.md
+---
+# 장비
+
+* 테스트폰 갤럭시 S24 1대: [이민준](/people/minjun.md)이 11월 15일까지 빌려 감(심사용 빌드 확인).[^07_equipment_memo]
+* 테스트폰 아이폰 15 1대: 비어 있음.[^07_equipment_memo]
+* 태블릿 아이패드 에어 1대: [강도윤](/people/doyun.md)이 쓰는 중(원화 작업용).[^07_equipment_memo]
+* 녹음용 마이크 1대: 비어 있음.[^07_equipment_memo]
+
+[^07_equipment_memo]: 07_equipment_memo.md

===== studio/log.md
+* **Creation**: 07_equipment_memo.md를 반영해 [사내 장비 목록](/equipment/inventory.md) 문서를 만들었습니다.
~~~

**결과 읽기**
- **새 폴더:** `studio/equipment/`와 폴더 목차가 생기고, 번들 맨 위 목차(`studio/index.md`)에 `장비` 한 줄이 더해집니다. 문서 속 사람 이름은 사람 문서로 링크됩니다.
- **사람 확인:** 새 폴더는 위키의 구조를 바꾸는 일이라 답의 사람이 확인할 것에 그 위치를 적습니다. 폴더 이름(`equipment`)과 문서 `type`은 에이전트가 정한 값이라 사람이 확인합니다.

### 3-5. 새 번들

`raw/06_lore_memo.md`는 게임 세계관(주인공, 무대, 악역) 메모입니다. 업무 위키(사람, 프로젝트, 회의, 업무 절차, 결정)에도 데이터 위키(데이터셋, 테이블, 지표)에도 맞지 않습니다. 프로젝트 문서에 '초롱 원정대'라는 같은 이름이 나오지만 그 문서에는 세계관을 넣을 절이 없으므로, 기존 문서를 고치지 않고 새 번들을 만듭니다.

⌨️ 입력 (Codex)

~~~text
$okf-ingest raw/06_lore_memo.md
~~~

💬 답 (Codex)

~~~text
결과: 새 번들, `lore/world/chorong_expedition`, `draft`

사람이 확인할 것:

- 기존 번들의 설명 범위에 맞지 않는다고 판단해 `lore` 번들과 `world` 폴더를 만들고, 문서 유형을 `worldbuilding`으로 정했습니다.
- 필수 검사에서 기존 `studio/playbooks/server_incident.md`의 `stale_after: 2026-10-01`이 지났다고 보고됐습니다. 이 문서는 수정하지 않았습니다.

원문은 수정하지 않았습니다. 신규 문서에는 검사 문제가 없습니다.
~~~

새 번들의 파일입니다(모두 새 파일이라 모든 줄이 `+`).

~~~diff
===== lore/index.md
+---
+okf_version: "0.2"
+---
+# 초롱 원정대 세계관 위키
+
+* [세계관](world/) - 초롱 원정대 세계와 주요 설정

===== lore/log.md
+# 초롱 원정대 세계관 위키 변경 기록
+
+## 2026-11-02
+
+* **Initialization**: 번들을 만들었습니다.
+* **Creation**: 06_lore_memo.md를 반영해 [초롱 원정대 세계관](/world/chorong_expedition.md) 문서를 만들었습니다.

===== lore/world/index.md
+# 세계관
+
+* [초롱 원정대 세계관](chorong_expedition.md) - 초롱이의 배경, 목표와 적대자

===== lore/world/chorong_expedition.md
+---
+type: worldbuilding
+title: 초롱 원정대 세계관
+description: 초롱이와 밤하늘 섬, 별등을 중심으로 한 원정대 세계관.
+tags: [lore, worldbuilding]
+status: draft
+generated: { by: "codex", at: "2026-11-02T17:20:22+09:00" }
+sources:
+  - id: 06_lore_memo
+    resource: "raw://06_lore_memo.md"
+    title: "06_lore_memo.md"
+---
+# 초롱 원정대 세계관
+
+## 배경
+
+초롱이는 반딧불 마을에서 태어난 등불 요정이다.[^06_lore_memo]
+
+밤하늘 섬은 모두 일곱 개이며, 섬마다 꺼진 별등이 하나씩 있다.[^06_lore_memo]
+
+## 목표
+
+초롱이의 목표는 일곱 별등을 다시 밝혀 밤하늘을 되찾는 것이다.[^06_lore_memo]
+
+## 적대자
+
+빛을 먹는 그림자 고래 '먹구름'이 악역이다.[^06_lore_memo]
+
+[^06_lore_memo]: 06_lore_memo.md
~~~

**결과 읽기**
- **새 번들:** 맨 위 목차(`okf_version`), 변경 기록(Initialization, Creation), 폴더와 폴더 목차, 문서까지 OKF 번들의 모양을 갖춰 만들어집니다. 검사 스크립트와 부록 02의 비서는 `index.md`가 있는 폴더를 모두 번들로 보므로, 새 번들도 코드를 바꾸지 않고 함께 다룹니다.
- **규칙이 바로잡은 판단:** 이 부록을 만들 때 넣기 스킬에 '들어갈 절이 없으면 기존 문서를 고치지 않는다' 규칙을 넣기 전에는, 에이전트가 프로젝트 문서에 `## 세계관` 절을 새로 만들어 넣었습니다. 같은 이름이 나온다는 이유로 문서를 고른 것입니다. 에이전트가 같은 판단을 되풀이해 틀리면, 그 판단을 막는 규칙을 스킬에 적습니다.
- **사람 확인:** 번들 이름(`lore`)과 문서 `type`(`worldbuilding`)은 에이전트가 정한 값이라 사람이 확인합니다.

### 3-6. 위키 점검

점검은 문서를 고치지 않습니다.

⌨️ 입력 (Claude Code)

~~~text
/okf-lint
~~~

⌨️ 입력 (Codex)

~~~text
$okf-lint
~~~

💬 답 (Codex, 3-2부터 3-5까지 일곱 건을 넣은 뒤)

~~~text
점검 시각: 2026-11-02 17:25 KST. `check.py`에서 문제 1건을 보고했습니다.

| 번들 | 점검 | 문서 | 내용 |
|---|---|---|---|
| studio | 만료 | server_incident.md | `stale_after: 2026-10-01`이 지났습니다. |

| 문서 A의 줄 | 문서 B의 줄 | 모순 |
|---|---|---|
| bundles/studio/projects/chorong_launch.md:26, 정식 출시 12월 4일 | bundles/studio/playbooks/launch_checklist.md:12, 정식 출시 11월 20일 | 정식 출시일이 서로 다릅니다. |

사람이 정할 것:

- `server_incident` 내용을 누가 다시 확인할지 정하고, 확인 결과에 따라 문서의 만료 시점과 내용을 갱신합니다.
- 출시일의 기준 문서를 정하고, 다른 문서의 날짜를 맞춥니다.

문서는 수정하지 않았습니다.
~~~

**결과 읽기**
- **검사 결과:** `check.py`가 찾은 문제(서버 장애 대응 문서의 만료)를 표로 옮깁니다. 일곱 건을 넣는 동안 답마다 알리던 그 문제이고, 이번 자료와는 관계없이 원래 위키에 있던 문제입니다.
- **모순:** 3-2에서 고친 프로젝트 문서의 출시일(12월 4일)과, 고치지 않은 출시 체크리스트의 출시일(11월 20일)이 다르다고 알립니다. 어느 쪽을 고칠지는 사람이 정하고, 고치라고 지시하면 에이전트가 넣기 스킬로 고칩니다.
- **따라 할 때 다를 수 있는 점:** 에이전트가 3-2에서 체크리스트까지 함께 고쳤다면 모순이 나오지 않습니다. 실행마다 판단이 조금씩 다르므로, 답의 사람이 확인할 것과 바뀐 줄을 함께 봅니다.

## 4. 핵심 정리

- **스킬 구성:** 스킬 하나는 `SKILL.md`(frontmatter와 본문)와 스크립트를 담은 폴더입니다. 저장 규칙은 본문에 적고, 형식은 검사 스크립트(`check.py`)로 확인합니다.
- **스킬 등록:** Claude Code는 `.claude/skills/`, Codex는 `.agents/skills/`에서 스킬을 찾으므로 같은 스킬을 두 폴더에 둡니다. 파일을 고칠 때 Claude Code는 스킬의 `allowed-tools`에 적은 도구를 묻지 않고 쓰고, Codex는 처음 실행할 때 폴더 사용을 허용받아 작업 폴더 안 파일을 고칩니다.
- **스킬 활용:** 지시가 스킬 설명과 맞으면 에이전트가 스스로 고르고, `/이름`(Claude Code), `$이름`(Codex)으로 직접 부를 수도 있습니다. 에이전트의 판단이 문서에 바로 들어가므로, 답의 사람이 확인할 것과 바뀐 줄은 사람이 확인합니다.
- **일반화:** 넣을 번들은 각 번들의 맨 위 목차를 읽고 고르고, 맞는 번들이 없으면 새 번들을 만듭니다. 그래서 스킬은 위키의 번들과 문서가 바뀌어도 그대로 씁니다.